# RandomForestRegressor → fail-closed decision model

**Problem.** Predict disease progression (`diabetes`, all 10 features) and decide whether the prediction is
*safe enough to act on* at a declared threshold of **240**. `sklearn`'s diabetes features are
mean-centred and scaled, so all thresholds below are in standardized units.

**Why a harness, not just a model.** A regressor always returns a number. A decision model must answer
*"may this run?"* — and must refuse when the number is not trustworthy. `mjolnir_ml` wraps the fitted
RandomForest with:

- a **regime floor** — inputs outside the training support abstain;
- a **conformal κ** — an interval with finite-sample coverage at α = 0.10;
- a **declared policy** (`@dsl`) over the interval: release only if the whole interval clears the band,
  deny only if the whole interval is above it, ask otherwise;
- a **certificate** — claim/policy/model/calibration digests, replayable without the model.

**How to read the DSL.** Statements see two kinds of facts: *stated input facts* (features such as `bmi`)
and *model facts* supplied by the harness (`pred`, `lo`, `hi`). A **premise** that fails makes
the row undecidable (`abstain`); a **rule** that fires escalates (`ask`) or refuses (`deny`); a **release**
statement is the only way to release.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # the mjolnir_ml package

import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ASK, DENY, RELEASE, DecisionModel, DecisionPolicyBase,
                        HazardLedger, dsl, report, rule, certify, verify_certificate)

pd.set_option("display.width", 140)

## 1. The learner and the data (all features)

All 10 features are kept: the forest sees the full representation and the policy may read any of them as
stated facts. `Xcal` is a **held-out calibration split** — conformal calibration must not see training
rows.

In [2]:
data = load_diabetes()
FEATURES = tuple(data.feature_names)
print("features:", FEATURES)
print("n =", len(data.data), " target range:", data.target.min(), "-", data.target.max())

Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.4, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5, random_state=0)

rows = lambda A: [dict(zip(FEATURES, map(float, x))) for x in A]
train_rows, cal_rows, test_rows = rows(Xtr), rows(Xcal), rows(Xte)

forest = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
pd.Series(forest.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

features: ('age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6')
n = 442  target range: 25.0 - 346.0


s5     0.363
bmi    0.239
bp     0.105
s6     0.057
age    0.055
s3     0.054
s1     0.045
s2     0.045
s4     0.022
sex    0.015
dtype: float64

## 2. The policy (compiled from source)

- `bmi_in_range` — a **premise** (sanity bound on the standardized BMI): a void input cannot be decided.
- `release_safe_band` — the *entire* conformal interval is below 240 → `release`.
- `deny_high_band` — the *entire* interval is above 240 → `deny` (nobody may grant this).
- `review_widening` — the interval straddles 240 → `ask` (a human decides).

The policy never sees the forest object; it sees `lo`/`hi` (conformal) and stated facts.

In [3]:
@dsl
class ProgressionPolicy(DecisionPolicyBase):
    facts = ("age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6")

    assert lambda bmi: bmi > -1.0, "bmi_in_range"

    assert rule("release_safe_band",
                when=lambda lo, hi: lo >= 0.0 and hi <= 240.0, verdict=RELEASE)
    assert rule("deny_high_band", when=lambda lo: lo > 240.0, verdict=DENY)
    assert rule("review_widening",
                when=lambda lo, hi: lo <= 240.0 and hi > 240.0, verdict=ASK)

ProgressionPolicy.policy()

Policy('ProgressionPolicy', 4 statements, sha256:48ff8bf11b01e03...)

## 3. Bind, floor, calibrate, dispose

`fit_regime` records the training support; `calibrate` fits the conformal index at α = 0.10. One row is then
disposed: verdict, basis, guarantee tag and certificate id are the product.

In [4]:
dm = (DecisionModel(forest, ProgressionPolicy, features=FEATURES)
      .fit_regime(train_rows)
      .calibrate(cal_rows, ycal, alpha=0.10))

d = dm.dispose(test_rows[0])
print("verdict :", d.verdict)
print("basis   :", list(d.basis))
print("say     :", d.say)
print("facts   :", {k: round(v, 3) for k, v in d.prediction.items() if isinstance(v, float)})
print("cert id :", d.certificate_id[:34], "…")
print("replay  :", dm.replay(d, test_rows[0]))

verdict : ask
basis   : ['review_widening']
say     : escalated by review_widening (lo=143.62666666666667, hi=360.94666666666666)
facts   : {'pred': 252.287, 'lo': 143.627, 'hi': 360.947}
cert id : sha256:90f068b8660dc489e3922b6db42 …
replay  : True


### Gates in action

The pipeline order is **contract → regime → predict → premises → rules**, and each gate fails closed.
Feed an implausible BMI and the *regime floor* refuses before the forest is called: the harness never
extrapolates outside the recorded training support. (Premises sit after the regime check and catch
states that are individually in-range but jointly invalid — e.g. `assert lambda bmi, bp: not (bmi > 0.15 and bp < 0.0)`.)

In [5]:
bad = dict(test_rows[0], bmi=-5.0)
d_bad = dm.dispose(bad)
print(d_bad.verdict, "|", list(d_bad.basis), "|", d_bad.say)

abstain | ['out_of_regime:bmi'] | cannot decide: bmi=-5 is outside recorded training bounds [-0.0848862, 0.170555]


## 4. The comprehensive report (problem · decisions · hazards · certificates)

`report()` is the single artifact for a reviewer: the compiled policy, verdict census, abstention causes,
the Ω pair against a pinned oracle (`safe = y <= 240`), completeness against an **external** hazard ledger,
and the certificate census. The ledger is external to the policy: spec limits on BMI and blood pressure,
declared with provenance.

In [6]:
ledger = (HazardLedger()
          .add("extreme_bmi", lambda r: r["bmi"] > 0.10, "clinical spec §2 (BMI)")
          .add("high_bp", lambda r: r["bp"] > 0.14, "clinical spec §3 (blood pressure)"))

safe = [int(v) <= 240 for v in yte]
decidable = [True] * len(yte)

text = report(dm, test_rows, title="Diabetes progression — safe-to-act decision",
              safe=safe, decidable=decidable, ledger=ledger,
              key=b"demo-audit-key", sample=10)
print(text)

 Diabetes progression — safe-to-act decision
 mjolnir.ml.report/1 · 2026-10-22T16:48:11Z

CONTRACT (the problem)
----------------------
  policy        ProgressionPolicy  (sha256:48ff8bf11b01e03ed946078fb8e51acb15ac1f18deeb34d86d070816e068dcbf)
  model         RandomForestRegressor  (sha256:b2fac7b2d0deb7bf…)
  calibration   alpha=0.10  (sha256:16d4215fb5aeed59…)
  regime        required   features=10
  features      age, sex, bmi, bp, s1, s2, s3, s4, s5, s6

POLICY (every statement, compiled)
----------------------------------
  code               kind     requires  predicate                   verdict
  -----------------  -------  --------  --------------------------  -------
  bmi_in_range       premise  bmi       bmi > -1.0                  -      
  release_safe_band  release  lo, hi    lo >= 0.0 and hi <= 240.0   release
  deny_high_band     rule     lo        lo > 240.0                  deny   
  review_widening    rule     lo, hi    lo <= 240.0 and hi > 240.0  ask    

DECISIONS

## 5. Auditing one decision without the forest

The certificate publishes the fact table. An auditor with the policy artifact and the key re-derives the
verdict with `Policy.dispose_facts` — no RandomForest, no training data. Tampering is rejected.

In [7]:
cert = certify(dm.dispose(test_rows[0]), key=b"demo-audit-key")
print("verified (key + policy only):",
      verify_certificate(cert, key=b"demo-audit-key", policy=ProgressionPolicy))
print("replayable from facts      :", cert["replayable"])

bad_cert = dict(cert, verdict="release" if cert["verdict"] != "release" else "deny")
try:
    verify_certificate(bad_cert, key=b"demo-audit-key", policy=ProgressionPolicy)
except Exception as ex:
    print("tampered verdict rejected  :", type(ex).__name__, "—", ex)

verified (key + policy only): True
replayable from facts      : True
tampered verdict rejected  : CertificateError — signature does not match the certificate


## Takeaways

- The forest is a **proposal of facts** (`pred`, conformal `lo`/`hi`); the policy is the only disposer.
- A release requires the *whole* interval to clear the band. Wide intervals become `ask` — the price of
  soundness is visible in the report.
- The report is digest-pinned and the certificate replays without the learner.
- Swap `RandomForestRegressor` for `Ridge` or `HistGradientBoostingRegressor`: policy, κ and certificates
  are unchanged; only `model_digest` moves.